Reference function

In [1]:
def make_dataset(
    couples_df: pd.DataFrame,
    bacteria_model_names: List[str],
    phages_model_names: List[str],
    output_manager: EmbeddingsManager,
    device: str,
) -> pd.DataFrame:
    """
    Create a dataset with embeddings for bacteria and phages by loading them from the output manager. The final DataFrame will have columns: bacterium_id, phage_id, interaction_type, bacterium_embedding, phage_embedding.

    :param couples_df: DataFrame containing the couples data with columns `bacterium_id`, `phage_id` and `interaction_type`.
    :type couples_df: pd.DataFrame
    :param bacteria_model_names: List of bacteria embedding model names to load embeddings from. Used to load the corresponding embeddings for each bacterium.
    :type bacteria_model_names: List[str]
    :param phages_model_names: List of phage embedding model names to load embeddings from.
    :type phages_model_names: List[str]
    :param output_manager: EmbeddingsManager instance to handle loading of embeddings.
    :type output_manager: EmbeddingsManager
    :param device: Device to load the embeddings onto (e.g., "cpu" or "cuda:0").
    :type device: str
    :return: DataFrame with columns: bacterium_id, phage_id, interaction_type, bacterium_embedding, phage_embedding.
    :rtype: DataFrame
    """

    # Helper function to average tensors of different lengths. Not currently used, but kept for reference.
    def avg_tensors(sequences):
        import torch.nn.functional as F

        num = len(sequences)
        max_len = max([s.size(0) for s in sequences])
        out_dims = (num, max_len)
        out_tensor = sequences[0].data.new(*out_dims).fill_(0)
        mask = sequences[0].data.new(*out_dims).fill_(0)
        for i, tensor in enumerate(sequences):
            length = tensor.size(0)
            out_tensor[i, :length] = tensor
            mask[i, :length] = 1

        return torch.mean(out_tensor, dim=0)

    result = couples_df.copy(deep=True)

    logger.info(f"Creating dataset (loading embeddings)...")

    bacteria_embeddings = []
    for bacteria_model in bacteria_model_names:
        bacteria_embeddings.append(
            output_manager.load_embedding_batch(
                result["bacterium_id"].tolist(),
                model_name=bacteria_model,
                device=device,
            )
        )

    # The embeddings are concatenated to form 1 final embedding per bacterium/phage.
    result["bacterium_embedding"] = pd.Series(
        [torch.cat(embeds) for embeds in zip(*bacteria_embeddings)]
    )
    # One of the papers mentions that you can also simply add them, to reduce the final size, but did not obtain better results
    # result["bacterium_embedding"] = pd.Series([avg_tensors(embeds) for embeds in zip(*bacteria_embeddings)]) # Avg of the embeddings instead of concat

    phage_embeddings = []
    for phage_model in phages_model_names:
        phage_embeddings.append(
            output_manager.load_embedding_batch(
                result["phage_id"].tolist(), model_name=phage_model, device=device
            )
        )
    result["phage_embedding"] = pd.Series(
        [torch.cat(embeds) for embeds in zip(*phage_embeddings)]
    )
    # result["phage_embedding"] = pd.Series([avg_tensors(embeds) for embeds in zip(*phage_embeddings)]) # Avg of the embeddings instead of concat

    # logger.debug(
    #     f"Final embedding size (bacteria): {result['bacterium_embedding']}"
    # )
    logger.debug(
        f"Final embedding size (bacteria): {len(result['bacterium_embedding'].iloc[0])}"
    )
    logger.debug(
        f"Final embedding size (phages): {len(result['phage_embedding'].iloc[0])}"
    )

    return result

NameError: name 'pd' is not defined

Modified function to see the sizes and order of the concatination

In [8]:
def make_dataset(
    couples_df: pd.DataFrame,
    bacteria_model_names: List[str],
    phages_model_names: List[str],
    output_manager: EmbeddingsManager,
    device: str,
) -> pd.DataFrame:
    """
    Create a dataset with embeddings for bacteria and phages by loading them from the
    output manager.

    This version:
    - Keeps one column per model for both bacteria and phages
      (e.g. `bacterium_embedding_NT2`, `phage_embedding_MegaDNA`, ...).
    - Also creates concatenated meta-embeddings
      (`bacterium_embedding`, `phage_embedding`) for compatibility.
    """

    # Helper function to average tensors of different lengths. Not currently used, but kept for reference.
    def avg_tensors(sequences):
        import torch.nn.functional as F

        num = len(sequences)
        max_len = max([s.size(0) for s in sequences])
        out_dims = (num, max_len)
        out_tensor = sequences[0].data.new(*out_dims).fill_(0)
        mask = sequences[0].data.new(*out_dims).fill_(0)
        for i, tensor in enumerate(sequences):
            length = tensor.size(0)
            out_tensor[i, :length] = tensor
            mask[i, :length] = 1

        return torch.mean(out_tensor, dim=0)

    result = couples_df.copy(deep=True)

    logger.info("Creating dataset (loading embeddings, per-model and concatenated)...")

    # -------------------------
    # BACTERIA: per-model cols
    # -------------------------
    bacteria_embeddings_per_model: List[List[torch.Tensor]] = []

    for bacteria_model in bacteria_model_names:
        embeds = output_manager.load_embedding_batch(
            result["bacterium_id"].tolist(),
            model_name=bacteria_model,
            device=device,
        )
        bacteria_embeddings_per_model.append(embeds)

        # One column per model, e.g. bacterium_embedding_NT2
        col_name = f"bacterium_embedding_{bacteria_model}"
        result[col_name] = pd.Series(embeds)

        # Optional: you can inspect lengths per model directly:
        # lengths = result[col_name].apply(lambda t: t.numel())
        # logger.debug(f"{col_name} length stats:\n{lengths.describe()}")

    # Concatenated bacteria meta-embedding (same as original implementation)
    result["bacterium_embedding"] = pd.Series(
        [torch.cat(embeds) for embeds in zip(*bacteria_embeddings_per_model)]
    )
    # If you ever want the average instead of concat:
    # result["bacterium_embedding"] = pd.Series(
    #     [avg_tensors(embeds) for embeds in zip(*bacteria_embeddings_per_model)]
    # )

    # ----------------------
    # PHAGE: per-model cols
    # ----------------------
    phage_embeddings_per_model: List[List[torch.Tensor]] = []

    for phage_model in phages_model_names:
        embeds = output_manager.load_embedding_batch(
            result["phage_id"].tolist(),
            model_name=phage_model,
            device=device,
        )
        phage_embeddings_per_model.append(embeds)

        # One column per model, e.g. phage_embedding_NT2
        col_name = f"phage_embedding_{phage_model}"
        result[col_name] = pd.Series(embeds)

        # Optional: inspect lengths:
        # lengths = result[col_name].apply(lambda t: t.numel())
        # logger.debug(f"{col_name} length stats:\n{lengths.describe()}")

    # Concatenated phage meta-embedding (same as original implementation)
    result["phage_embedding"] = pd.Series(
        [torch.cat(embeds) for embeds in zip(*phage_embeddings_per_model)]
    )
    # Or average instead of concat:
    # result["phage_embedding"] = pd.Series(
    #     [avg_tensors(embeds) for embeds in zip(*phage_embeddings_per_model)]
    # )

    logger.debug(
        f"Final concatenated embedding size (bacteria): "
        f"{len(result['bacterium_embedding'].iloc[0])}"
    )
    logger.debug(
        f"Final concatenated embedding size (phages): "
        f"{len(result['phage_embedding'].iloc[0])}"
    )

    return result

staritng here

In [2]:
import os
import sys

# 1. Change the working directory to the project root
%cd ..

# 2. Add the root directory to the Python path so imports work
sys.path.append(os.getcwd())

# 3. Verify we are in the right place
print(f"Current Working Directory: {os.getcwd()}")
# !ls # Optional: list files to confirm you see 'main.py' and 'pbi_utils'

/data/pavel.degterev/pbi
Current Working Directory: /data/pavel.degterev/pbi


In [3]:
from __future__ import annotations
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import classification_report

# Custom project imports
from pbi_utils.config_parser import parse_config
from pbi_utils.data_manager import PerphectDataInput, H5pyEmbeddingsManager
from main import (
    #make_dataset,
    train_model,
    test_model,
    compute_metrics,
    logger,
)

%matplotlib inline

In [4]:
config_path = "model_configs/best_model.yaml"
config = parse_config(config_path=config_path)
device = config.device

# 1. Load CSV data
bacteria_df, phages_df, couples_df = PerphectDataInput(input_paths=config.input_perphect).load()

# 2. Load Embeddings
output_manager = H5pyEmbeddingsManager(config.embeddings_dir)
bacteria_model_names = [m.name() for m in config.bacteria_embedding_models]
phages_model_names = [m.name() for m in config.phages_embedding_models]

# 3. Create Dataset
dataset = make_dataset(
    couples_df=couples_df,
    bacteria_model_names=bacteria_model_names,
    phages_model_names=phages_model_names,
    output_manager=output_manager,
    device=device,
)

[DEBUG] [NT2] Using InstaDeepAI default model weights


/home/pavel.degterev/miniforge3/envs/pbi/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


[DEBUG] [NT2] Max sequence length for Nucleotide Transformer: 12276
[DEBUG] [MegaDNA] Max sequence length for megaDNA: 131071
[DEBUG] [DNABERT2] Max sequence length for DNABERT2: 32768
[DEBUG] [NT2] Using InstaDeepAI default model weights
[DEBUG] [NT2] Max sequence length for Nucleotide Transformer: 12276
[DEBUG] [MegaDNA] Max sequence length for megaDNA: 131071
[DEBUG] [DNABERT2] Max sequence length for DNABERT2: 32768
[INFO] Configuration loaded from model_configs/best_model.yaml: Config(input_perphect=bacteria_df='data/perphect-data/all/bacteria_df.csv' phages_df='data/perphect-data/all/phages_df.csv' couples_df='data/perphect-data/all-private-oversampled/couples_df.csv', embeddings_dir=data/embeddings, num_gpu=1, gpu_id=0, training_config=TrainingConfig(do_train=True epochs=100 batch_size=256 learning_rate=0.001 weight_decay=0.0001 k_folds_cv=10 patience_early_stopping=1000 monitor_metric_early_stopping='f1' patience_reduce_lr=1000 monitor_metric_reduce_lr='f1' multiplying_factor_r

NameError: name 'make_dataset' is not defined

In [14]:
# After calling make_dataset(...)
for m in bacteria_model_names:
    col = f"bacterium_embedding_{m}"
    print(col, dataset[col].apply(lambda t: t.numel()).describe())

for m in phages_model_names:
    col = f"phage_embedding_{m}"
    print(col, dataset[col].apply(lambda t: t.numel()).describe())

bacterium_embedding_NT2-TopBottomTruncateStrategy-250M-ov0 count    10300.0
mean      1536.0
std          0.0
min       1536.0
25%       1536.0
50%       1536.0
75%       1536.0
max       1536.0
Name: bacterium_embedding_NT2-TopBottomTruncateStrategy-250M-ov0, dtype: float64
bacterium_embedding_MegaDNA-BottomTruncateStrategy-concat-ov0 count    10300.0
mean       964.0
std          0.0
min        964.0
25%        964.0
50%        964.0
75%        964.0
max        964.0
Name: bacterium_embedding_MegaDNA-BottomTruncateStrategy-concat-ov0, dtype: float64
bacterium_embedding_DNABERT2-TopBottomTruncateStrategy-ov0-maxlen32768 count    10300.0
mean      1536.0
std          0.0
min       1536.0
25%       1536.0
50%       1536.0
75%       1536.0
max       1536.0
Name: bacterium_embedding_DNABERT2-TopBottomTruncateStrategy-ov0-maxlen32768, dtype: float64
phage_embedding_NT2-TopBottomTruncateStrategy-250M-ov0 count    10300.0
mean      1536.0
std          0.0
min       1536.0
25%       1536.0
50

In [13]:
dataset.info()   

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10300 entries, 0 to 10299
Data columns (total 12 columns):
 #   Column                                                                  Non-Null Count  Dtype 
---  ------                                                                  --------------  ----- 
 0   id                                                                      10300 non-null  int64 
 1   phage_id                                                                10300 non-null  int64 
 2   bacterium_id                                                            10300 non-null  int64 
 3   interaction_type                                                        10300 non-null  int64 
 4   bacterium_embedding_NT2-TopBottomTruncateStrategy-250M-ov0              10300 non-null  object
 5   bacterium_embedding_MegaDNA-BottomTruncateStrategy-concat-ov0           10300 non-null  object
 6   bacterium_embedding_DNABERT2-TopBottomTruncateStrategy-ov0-maxlen32768  10300 non-null

In [15]:
bacteria_model_names

['NT2-TopBottomTruncateStrategy-250M-ov0',
 'MegaDNA-BottomTruncateStrategy-concat-ov0',
 'DNABERT2-TopBottomTruncateStrategy-ov0-maxlen32768']

In [16]:
phages_model_names

['NT2-TopBottomTruncateStrategy-250M-ov0',
 'MegaDNA-MaxStrategy-concat-ov0',
 'DNABERT2-TKPert-concat-J16-g20-ov0-maxlen32768']

In [17]:
import torch
import numpy as np

for m in bacteria_model_names:
    col = f"bacterium_embedding_{m}"
    tensors = dataset[col].dropna()
    
    # Stack all tensors into a 2D matrix: (n_samples, embedding_dim)
    mat = torch.stack(list(tensors)).float()  # (N, D)
    
    print(f"\n{'='*50}")
    print(f"[BACTERIA] Model: {m}")
    print(f"  Embedding dim : {mat.shape[1]}")
    print(f"  Samples       : {mat.shape[0]}")
    print(f"  Value min     : {mat.min().item():.6f}")
    print(f"  Value max     : {mat.max().item():.6f}")
    print(f"  Value mean    : {mat.mean().item():.6f}")
    print(f"  Value std     : {mat.std().item():.6f}")
    print(f"  Per-dim std   : min={mat.std(dim=0).min().item():.6f}, "
                           f"max={mat.std(dim=0).max().item():.6f}, "
                           f"mean={mat.std(dim=0).mean().item():.6f}")

for m in phages_model_names:
    col = f"phage_embedding_{m}"
    tensors = dataset[col].dropna()
    
    mat = torch.stack(list(tensors)).float()
    
    print(f"\n{'='*50}")
    print(f"[PHAGE] Model: {m}")
    print(f"  Embedding dim : {mat.shape[1]}")
    print(f"  Samples       : {mat.shape[0]}")
    print(f"  Value min     : {mat.min().item():.6f}")
    print(f"  Value max     : {mat.max().item():.6f}")
    print(f"  Value mean    : {mat.mean().item():.6f}")
    print(f"  Value std     : {mat.std().item():.6f}")
    print(f"  Per-dim std   : min={mat.std(dim=0).min().item():.6f}, "
                           f"max={mat.std(dim=0).max().item():.6f}, "
                           f"mean={mat.std(dim=0).mean().item():.6f}")


[BACTERIA] Model: NT2-TopBottomTruncateStrategy-250M-ov0
  Embedding dim : 1536
  Samples       : 10300
  Value min     : -3.392043
  Value max     : 10.644578
  Value mean    : 0.001087
  Value std     : 0.397159
  Per-dim std   : min=0.048357, max=1.013273, mean=0.156148

[BACTERIA] Model: MegaDNA-BottomTruncateStrategy-concat-ov0
  Embedding dim : 964
  Samples       : 10300
  Value min     : -3.136655
  Value max     : 3.576223
  Value mean    : 0.008602
  Value std     : 0.291290
  Per-dim std   : min=0.000004, max=1.113464, mean=0.080614

[BACTERIA] Model: DNABERT2-TopBottomTruncateStrategy-ov0-maxlen32768
  Embedding dim : 1536
  Samples       : 10300
  Value min     : -4.845669
  Value max     : 2.118389
  Value mean    : 0.002865
  Value std     : 0.133699
  Per-dim std   : min=0.017566, max=1.161894, mean=0.053213

[PHAGE] Model: NT2-TopBottomTruncateStrategy-250M-ov0
  Embedding dim : 1536
  Samples       : 10300
  Value min     : -3.346185
  Value max     : 10.745193
  Val

In [18]:
import torch
import matplotlib.pyplot as plt

def plot_block_distribution(tensors, title, bins=100, value_range=None, logy=True):
    # tensors: pd.Series of 1D torch tensors
    mat = torch.stack(list(tensors)).float()   # (N, D)
    vals = mat.flatten().cpu().numpy()

    plt.figure(figsize=(6, 4))
    plt.hist(vals, bins=bins, range=value_range, density=True, alpha=0.8, color="tab:blue")
    if logy:
        plt.yscale("log")
    plt.title(title)
    plt.xlabel("Embedding value")
    plt.ylabel("Density")
    plt.tight_layout()
    plt.show()